# 01. Quick Baseline

당일 **가장 빠르게 baseline 제출**을 만드는 노트북입니다.
src 모듈을 그대로 호출해 정형 파이프라인을 돌립니다.

## STEP 0. 경로 설정 (src import)

In [ ]:
import sys, os
sys.path.append('..')   # repo 루트를 path에 추가
import numpy as np
from src.utils.seed import seed_everything
from src.utils.metrics import get_metric, metric_name
seed_everything(42)
print('ready')

## STEP 1. config 로드 & 데이터

In [ ]:
import yaml
cfg = yaml.safe_load(open('../configs/tabular.yaml'))
cfg['task'] = 'binary'
print('task:', cfg['task'], '| folds:', cfg['n_folds'])

from src.data import loaders
X, y, groups, X_test, test_ids = loaders.load_tabular(cfg, demo=True)
print('train:', X.shape, '| test:', X_test.shape)

## STEP 2. 부스팅 3종 학습 (OOF 생성)

In [ ]:
from src.data.cv import get_folder, split as cv_split
from src.models.tabular import make_models

task = cfg['task']
folder = get_folder(task, cfg['n_folds'], cfg['seed'])

results = {}
oofs = {}
for m in make_models(task):
    oof = np.zeros(len(X))
    for tr_idx, va_idx in cv_split(folder, X.values, y):
        mk = [x for x in make_models(task) if x.kind == m.kind][0]
        mk.fit(X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx], y[va_idx])
        oof[va_idx] = mk.predict(X.iloc[va_idx])
    score = get_metric(task, y, oof)
    results[m.kind] = score; oofs[m.kind] = oof
    print(f'[{m.kind}] OOF {metric_name(task)}: {score:.5f}')

## STEP 3. 앙상블 (단순 평균)

In [ ]:
blend = np.mean(list(oofs.values()), axis=0)
print(f'앙상블 OOF {metric_name(task)}: {get_metric(task, y, blend):.5f}')
print(f'최고 단일 모델: {max(results.values()):.5f}')
print('\n→ 가중치 최적화는 src.ensemble 사용')

## STEP 4. 다음 단계

```bash
# CLI로 전체 파이프라인 (학습→추론→제출)
python -m src.train --config configs/tabular.yaml --exp exp_001 --demo
python -m src.ensemble --task binary --exps exp_001 exp_002
bash scripts/make_submission.sh configs/tabular.yaml exp_001
```